In [101]:
import numpy as np

In [102]:
import pandas as pd

course = pd.read_csv("Courses.csv")

### index 컬럼 제거

- `index` 컬럼은 단순한 행 번호(row number)로, 분석에 유의미한 정보를 포함하지 않음
- 설명 변수 또는 타겟 변수로 활용 가치가 없다고 판단

→ 불필요한 컬럼이므로 전처리 단계에서 제거

In [103]:
course = course.drop(columns=["index"])

### roles 컬럼 제거

- `roles` 컬럼은 전체 데이터에서 모두 결측(null)으로 확인됨
- 해당 변수는 어떠한 정보도 제공하지 않으며,
  분석에 활용 가능한 의미를 가지지 않음

처리 방식:
- 정보가 없는 컬럼으로 판단하여 제거

→ 불필요한 변수 제거를 통해 데이터 구조를 간결하게 정리

In [104]:
course = course.drop(columns=["roles"])

### explored, certified 컬럼 정합성 검증 및 전처리

#### 개념 정의
- `explored`는 강의 코스웨어의 50% 이상에 접근한 학습자 여부 (0/1)
- 퍼널 상 `viewed` 이후 단계에 해당
→ 즉, `explored = 1`이면 반드시 `viewed = 1`이 선행되어야 함

#### 전처리 기준 (퍼널 정합성)
- 정상 흐름: `viewed → explored`
- 이 순서를 위반하는 데이터는 논리적으로 불가능한 케이스로 판단

#### 제거 대상
- `viewed = 0` & `explored = 1` → 8건
- 한 번도 열람하지 않고 50% 이상 학습하는 것은 불가능
→ 데이터 오류로 판단하여 제거

- `certified`는 행동 단계가 아닌 결과(성과) 변수로 판단
→ 퍼널에서 제외하고 보조 지표로 활용
- `explored = 0` & `certified = 1` → 약 690건
- 유지 결정

- `certified`가 퍼널에서 제외되었기 때문에 퍼널 분석에는 영향 없음
- 수료증만을 목표로 한 학습자 집단으로 해석 가능
→ 별도 분석 가치 존재

#### 최종 결론
- 퍼널 위반(논리적 오류) 데이터만 제거
- 행동 정의와 충돌하지 않는 데이터는 유지

In [105]:
# 제거 대상만 따로 보기
invalid_funnel = course[
    ((course["explored"] == 1) & (course["viewed"] != 1))
]

len(invalid_funnel)

8

In [106]:
course = course[
    ~(
        ((course["explored"] == 1) & (course["viewed"] != 1))
    )
]

In [107]:
removed_count = course.shape[0] - course.shape[0]
print("Removed rows:", removed_count)

Removed rows: 0


### incomplete_flag 데이터 제거 (심화 근거)

#### 문제 정의
- incomplete_flag = 1인 데이터는 특정 변수에서 결측이 집중적으로 발생
- 특히 n_events는 대부분 결측이며, 다른 변수 또한 분포 차이를 보임

#### 결측 메커니즘 관점 해석
- 결측이 특정 그룹(incomplete_flag)에 집중되어 나타남
→ 이는 Missing Completely at Random(MCAR)이 아닌,
   **Missing at Random(MAR) 또는 Missing Not at Random(MNAR)** 구조로 판단됨

- 이러한 경우, 결측 자체가 데이터의 특성과 관련될 가능성이 높으며,
  분석 결과에 편향(bias)을 유발할 수 있음

#### 샘플 대표성 문제 (Selection Bias)
- incomplete_flag = 1 데이터를 포함할 경우,
  일부 사용자 그룹의 행동이 과소 또는 왜곡 반영될 가능성이 있음

- 특히 행동 로그 기반 분석에서는,
  관측 가능한 집단과 관측 불가능 집단이 혼재될 경우
  전체 사용자 행동을 대표하지 못하는 문제가 발생함

#### 식별성(Identifiability) 문제
- 행동 로그가 일부만 존재하는 경우,
  해당 사용자의 실제 행동 상태를 정의할 수 없음

- 이는 퍼널 단계 정의 및 KPI 산출 과정에서
  해석 불가능한 상태를 초래함

#### 처리 결정
→ 결측이 구조적으로 발생하며 분석 편향을 유발할 가능성이 높다고 판단하여,
   데이터의 일관성과 해석 가능성을 확보하기 위해
   `incomplete_flag = 1` 데이터를 분석에서 제외
- 단순 결측 대체(imputation)는 고려했으나,
  n_events와 같이 구조적으로 결측이 발생한 변수의 경우
  값 보정이 실제 행동을 반영하지 못할 가능성이 높다고 판단

- 또한 incomplete_flag=1 데이터는 일부 변수만 관측된 상태로,
  사용자 행동을 완전하게 정의할 수 없기 때문에
  부분적 보정보다는 제거가 더 적절하다고 판단

#### 결론
- 본 제거는 단순 결측 처리의 문제가 아니라,
  **데이터 생성 구조와 결측 메커니즘을 고려한 분석 설계 결정**임

In [108]:
import pandas as pd
import numpy as np

# =========================
# 0. 활동 지표 컬럼 정의
# =========================
activity_cols = [
    "last_event_DI",
    "nevents",
    "ndays_act",
    "nplay_video",
    "nchapters",
    "nforum_posts"
]

# =========================
# 1. incomplete_flag 기본 분포 확인
# =========================
print("=== incomplete_flag 분포 ===")
print(course["incomplete_flag"].value_counts(dropna=False))
print()

print("=== incomplete_flag 비율(%) ===")
print((course["incomplete_flag"].value_counts(dropna=False, normalize=True) * 100).round(2))
print()

# =========================
# 2. incomplete_flag별 활동지표 결측 개수 / 결측 비율 확인
# =========================
print("=== incomplete_flag별 활동지표 결측 개수 ===")
missing_count = course.groupby("incomplete_flag")[activity_cols].apply(lambda x: x.isna().sum())
print(missing_count)
print()

print("=== incomplete_flag별 활동지표 결측 비율 ===")
missing_ratio = course.groupby("incomplete_flag")[activity_cols].apply(lambda x: x.isna().mean().round(4))
print(missing_ratio)
print()



=== incomplete_flag 분포 ===
incomplete_flag
NaN    540971
1.0    100159
Name: count, dtype: int64

=== incomplete_flag 비율(%) ===
incomplete_flag
NaN    84.38
1.0    15.62
Name: proportion, dtype: float64

=== incomplete_flag별 활동지표 결측 개수 ===
                 last_event_DI  nevents  ndays_act  nplay_video  nchapters  \
incomplete_flag                                                              
1.0                      85599   100159      63752        98163      27593   

                 nforum_posts  
incomplete_flag                
1.0                         0  

=== incomplete_flag별 활동지표 결측 비율 ===
                 last_event_DI  nevents  ndays_act  nplay_video  nchapters  \
incomplete_flag                                                              
1.0                     0.8546      1.0     0.6365       0.9801     0.2755   

                 nforum_posts  
incomplete_flag                
1.0                       0.0  



### incomplete_flag 분포 확인

- 전체 데이터 중 약 15.6%가 `incomplete_flag = 1`에 해당함
- 단순 소수 이상치가 아니라, 분석에 영향을 줄 수 있는 규모의 데이터로 판단됨

→ 따라서 해당 그룹에 대한 별도의 검토가 필요함

### 활동 지표 결측 개수 확인

- `nevents`: 전부 결측 (100%)
- `nplay_video`: 대부분 결측 (~98%)
- `last_event_DI`: 약 85% 결측
- `ndays_act`: 약 64% 결측
- `nchapters`: 약 28% 결측
- `nforum_posts`: 결측 없음

→ 변수별로 결측 패턴이 크게 다르게 나타남
→ 단순 누락이라기보다, 로그 수집 구조의 차이를 시사

- 특정 변수(`nevents`, `nplay_video`)는 거의 관측되지 않는 반면,
  다른 변수(`nforum_posts`)는 완전히 관측됨

→ 동일한 기준에서 생성된 데이터라면 발생하기 어려운 패턴

→ 이는 일부 로그만 선택적으로 기록된
   **부분 관측(partially observed) 데이터**로 해석됨

In [109]:
# =========================
# 3. incomplete_flag별 활동지표 기초통계 확인
#    (수치형 활동지표 위주)
# =========================
numeric_activity_cols = [
    "nevents",
    "ndays_act",
    "nplay_video",
    "nchapters",
    "nforum_posts"
]

print("=== incomplete_flag별 활동지표 기초통계 ===")
for col in numeric_activity_cols:
    print(f"\n--- {col} ---")
    print(course.groupby("incomplete_flag")[col].describe())

=== incomplete_flag별 활동지표 기초통계 ===

--- nevents ---
                 count  mean  std  min  25%  50%  75%  max
incomplete_flag                                           
1.0                0.0   NaN  NaN  NaN  NaN  NaN  NaN  NaN

--- ndays_act ---
                   count      mean        std  min  25%  50%  75%    max
incomplete_flag                                                         
1.0              36407.0  4.390282  10.003355  1.0  1.0  1.0  3.0  176.0

--- nplay_video ---
                  count       mean        std  min  25%  50%   75%    max
incomplete_flag                                                          
1.0              1996.0  12.216934  17.443228  1.0  2.0  7.0  15.0  327.0

--- nchapters ---
                   count     mean       std  min  25%  50%  75%   max
incomplete_flag                                                      
1.0              72566.0  1.79879  1.624602  1.0  1.0  1.0  2.0  34.0

--- nforum_posts ---
                    count      mean    

### 활동 지표 기초통계 (incomplete_flag = 1)

- `ndays_act`: 평균 약 4.39일 (활동 수준 낮음)
- `nplay_video`: 일부 관측되나 표본 수 매우 적음
- `nchapters`: 평균 약 1.8 (적은 학습 수준)
- `nforum_posts`: 거의 0 (상호작용 거의 없음)

→ 전반적으로 활동 수준이 낮고,
   변수별 관측 여부도 불균형하게 나타남

→ 정상적인 학습 행동을 반영한 데이터로 보기 어려움

In [110]:
# =========================
# 4. incomplete_flag별 퍼널 / 성과 변수 평균 비교
# =========================
compare_cols = ["viewed", "explored", "certified", "grade"]

print("\n=== incomplete_flag별 주요 변수 평균 ===")
print(course.groupby("incomplete_flag")[compare_cols].mean(numeric_only=True).round(4))
print()


=== incomplete_flag별 주요 변수 평균 ===
                 viewed  explored  certified
incomplete_flag                             
1.0              0.7413    0.0204     0.0001



### incomplete_flag별 행동 지표 비교

- viewed: 약 74% (상대적으로 높음)
- explored: 약 2% (매우 낮음)
- certified: 거의 0%

→ 강의 진입(view)은 있으나,
   실제 학습(explored)이나 성취(certified)로 이어지지 않음

→ 의미 있는 학습 행동이라기보다
   비활성/저품질 사용자 로그일 가능성 존재

In [111]:
# =========================
# 5. 제거 전 데이터 백업
# =========================
course_before_incomplete = course.copy()

print("=== 제거 전 행 수 ===")
print(len(course_before_incomplete))
print()

# =========================
# 6. incomplete_flag = 1 제거
# =========================
course_after_incomplete = course_before_incomplete[
    course_before_incomplete["incomplete_flag"] != 1
].copy()

print("=== 제거 후 행 수 ===")
print(len(course_after_incomplete))
print()

print("=== 제거된 행 수 ===")
print(len(course_before_incomplete) - len(course_after_incomplete))
print()

print("=== 제거 비율(%) ===")
removed_ratio = (len(course_before_incomplete) - len(course_after_incomplete)) / len(course_before_incomplete) * 100
print(round(removed_ratio, 2))
print()

# =========================
# 7. 제거 전후 주요 KPI 비교
# =========================
kpi_cols = ["viewed", "explored", "certified", "grade"]

before_kpi = course_before_incomplete[kpi_cols].mean(numeric_only=True).rename("before")
after_kpi = course_after_incomplete[kpi_cols].mean(numeric_only=True).rename("after")

kpi_compare = pd.concat([before_kpi, after_kpi], axis=1)
kpi_compare["diff"] = kpi_compare["after"] - kpi_compare["before"]

print("=== 제거 전후 주요 KPI 비교 ===")
print(kpi_compare.round(4))
print()

=== 제거 전 행 수 ===
641130

=== 제거 후 행 수 ===
540971

=== 제거된 행 수 ===
100159

=== 제거 비율(%) ===
15.62

=== 제거 전후 주요 KPI 비교 ===
           before   after    diff
viewed     0.6243  0.6026 -0.0217
explored   0.0619  0.0696  0.0077
certified  0.0276  0.0327  0.0051



### 데이터 제거 규모 확인

- 전체 데이터의 약 15.6% 제거
- 절대적인 수는 크지만,
  분석 불가능하거나 신뢰하기 어려운 데이터로 판단됨

→ 데이터 양보다 데이터 품질을 우선시하여 제거 결정

### 제거 전후 주요 KPI 비교

- viewed: 소폭 감소 (-2.17%)
- explored: 증가 (+0.77%)
- certified: 증가 (+0.51%)

→ 제거 이후 학습 관련 지표(explored, certified)가 개선됨

→ incomplete_flag 데이터가
   실제 학습 행동을 희석시키는 노이즈 역할을 했을 가능성 존재

In [112]:
# =========================
# 8. 제거 전후 활동지표 평균 비교
# =========================
activity_compare_cols = ["nevents", "ndays_act", "nplay_video", "nchapters", "nforum_posts"]

before_activity = course_before_incomplete[activity_compare_cols].mean(numeric_only=True).rename("before")
after_activity = course_after_incomplete[activity_compare_cols].mean(numeric_only=True).rename("after")

activity_compare = pd.concat([before_activity, after_activity], axis=1)
activity_compare["diff"] = activity_compare["after"] - activity_compare["before"]

print("=== 제거 전후 활동지표 평균 비교 ===")
print(activity_compare.round(4))
print()

# =========================
# 9. 전처리 반영
#    실제 분석용 데이터로 확정
# =========================
course = course_after_incomplete.copy()

# 필요하면 incomplete_flag 컬럼 자체 제거
course_clean = course.drop(columns=["incomplete_flag"])

print("=== 최종 데이터 shape ===")
print(course_clean.shape)

=== 제거 전후 활동지표 평균 비교 ===
                before     after    diff
nevents       431.0138  431.0138  0.0000
ndays_act       5.7103    5.8191  0.1087
nplay_video   114.8442  115.9721  1.1279
nchapters       3.6342    4.0641  0.4299
nforum_posts    0.0190    0.0224  0.0034

=== 최종 데이터 shape ===
(540971, 18)


### 제거 전후 활동 지표 비교

- ndays_act, nplay_video, nchapters 등 주요 활동 지표 증가

→ 제거 이후 평균 활동 수준이 상승

→ incomplete_flag 데이터가
   실제 활동을 반영하지 못하는 저품질 로그였음을 시사

### last_event_DI 기반 전처리

본 단계에서는 마지막 활동 시점(`last_event_DI`)을 기준으로
사용자의 활동 상태를 재정의하고,
시간 정합성을 만족하지 않는 데이터를 정제하였다.

---

#### 3.1 Registered-only 사용자 처리

- 일부 데이터는 등록 시점(`start_time_DI`) 이후
  추가적인 활동이 기록되지 않은 상태로 존재함

- `last_event_DI`가 결측이며 `viewed != 1`인 경우,
  등록 이후 별도의 학습 활동이 없었던 사용자로 해석

처리 방식:
- 활동 지표(`nevents`, `ndays_act`, `nplay_video`, `nchapters`, `nforum_posts`)를 0으로 설정
- `last_event_DI = start_time_DI`로 보정

→ 등록 후 즉시 이탈한 사용자로 정의

---

#### 3.2 Viewed-only 사용자 처리

- 일부 데이터는 `viewed = 1`로 기록되어 있으나,
  `last_event_DI` 및 활동 지표가 결측으로 존재

- 이는 최소한 강의에 접근한 이력은 있으나,
  구체적인 활동 로그가 관측되지 않은 경우로 해석됨

- 해당 데이터는 행동 발생 여부(`viewed`)는 명확하지만,
  활동 강도는 확인할 수 없는 **부분 관측 데이터**에 해당함

처리 방식:
- 활동 지표는 임의로 보정하지 않고 결측 상태를 유지
- 시간 정합성을 위해 `last_event_DI = start_time_DI`로 보정
- 필요 시 별도 플래그로 관리

→ 퍼널 상 "진입 후 이탈 사용자"로 해석하여 유지

#### 3.3 duration < 0 데이터 제거

- `last_event_DI < start_time_DI`인 경우 존재
- 이는 시간 순서가 역전된 비정상 데이터

처리 방식:
- `duration < 0`인 데이터 제거

→ 시간 정합성을 만족하는 데이터만 유지

---

#### 3.4 duration = 0 패턴 재검토

- `duration = 0`은 시작일과 마지막 활동일이 같은 날임을 의미하며,
  반드시 비정상적인 학습을 의미하지는 않음

- 온라인 강의 특성상, 당일에 집중적으로 학습하여
  `explored = 1` 또는 `certified = 1`에 도달하는 사례가 발생할 수 있음

- 초기에는 이를 비정상 데이터로 간주하여 제거를 고려하였으나,
  명확한 오류로 단정하기 어려워 제거하지 않기로 결정

처리 방식:
- 해당 데이터는 제거하지 않고 유지
- 필요 시 단기 완료 패턴으로 별도 분석 대상으로 활용

---

#### 추가 검토: duration과 ndays_act 관계

- `duration`은 전체 학습 기간(날짜 차이)을 의미하고,
  `ndays_act`는 실제 활동이 발생한 날짜 수를 의미함

- 두 변수는 서로 다른 개념(기간 vs 활동 빈도)을 측정하므로,
  직접 비교를 통해 이상치를 정의하는 것은 적절하지 않음

- 특히 동일 날짜 내 활동은 `duration = 0`으로 계산될 수 있어
  `ndays_act > duration`과 같은 패턴이 발생할 수 있음

→ 해당 관계를 기준으로 데이터 제거는 수행하지 않음

---

#### 결론

→ `last_event_DI`를 기준으로 사용자 행동을 재정의하고,
   시간 순서 정합성은 엄격히 검증하되,
   학습 패턴 다양성은 보존하는 방향으로 전처리를 수행함

In [113]:
# 3.1, 3.2 코드 (수정본)

# 복사본
course_fixed = course_clean.copy()

# 조건 정의
mask_last_missing = course_fixed["last_event_DI"].isna()
mask_viewed_missing = mask_last_missing & (course_fixed["viewed"] == 1)
mask_non_viewed_missing = mask_last_missing & (course_fixed["viewed"] != 1)

# 활동성 관련 컬럼 지정
activity_cols = ["nevents", "ndays_act", "nplay_video", "nchapters", "nforum_posts"]

# 1) viewed == 1 인 경우
# 접근 이력은 있으나 활동량은 확정할 수 없으므로 활동지표는 건드리지 않음
# last_event_DI만 start_time_DI로 보정
course_fixed.loc[mask_viewed_missing, "last_event_DI"] = (
    course_fixed.loc[mask_viewed_missing, "start_time_DI"]
)

# 필요 시 플래그 추가
course_fixed["viewed_missing_flag"] = 0
course_fixed.loc[mask_viewed_missing, "viewed_missing_flag"] = 1

# 2) viewed != 1 인 경우
# 등록만 하고 추가 활동이 없었다고 보고 활동지표 0 처리
course_fixed.loc[mask_non_viewed_missing, activity_cols] = 0
course_fixed.loc[mask_non_viewed_missing, "last_event_DI"] = (
    course_fixed.loc[mask_non_viewed_missing, "start_time_DI"]
)

# 확인
print("Total missing last_event_DI:", mask_last_missing.sum())
print("Viewed=1 among missing:", mask_viewed_missing.sum())
print("Viewed!=1 among missing:", mask_non_viewed_missing.sum())

print(
    course_fixed.loc[
        mask_last_missing,
        ["viewed", "start_time_DI", "last_event_DI", "viewed_missing_flag"] + activity_cols
    ].head()
)

Total missing last_event_DI: 93353
Viewed=1 among missing: 1807
Viewed!=1 among missing: 91546
    viewed start_time_DI last_event_DI  viewed_missing_flag  nevents  \
25       0    2012-09-04    2012-09-04                    0      0.0   
32       0    2012-12-23    2012-12-23                    0      0.0   
35       0    2012-09-05    2012-09-05                    0      0.0   
40       0    2012-08-27    2012-08-27                    0      0.0   
45       0    2012-09-17    2012-09-17                    0      0.0   

    ndays_act  nplay_video  nchapters  nforum_posts  
25        0.0          0.0        0.0             0  
32        0.0          0.0        0.0             0  
35        0.0          0.0        0.0             0  
40        0.0          0.0        0.0             0  
45        0.0          0.0        0.0             0  


In [114]:
# 3.3 duration 계산 및 음수 duration 제거
course_fixed["last_event_DI"] = pd.to_datetime(course_fixed["last_event_DI"])
course_fixed["start_time_DI"] = pd.to_datetime(course_fixed["start_time_DI"])
course_fixed["duration"] = (
    course_fixed["last_event_DI"] - course_fixed["start_time_DI"]
).dt.days

mask_negative_duration = course_fixed["duration"] < 0

print("duration < 0 개수:", mask_negative_duration.sum())

course_fixed = course_fixed.loc[~mask_negative_duration].copy()

duration < 0 개수: 1341


In [115]:
# 3.4 차후 분석에 이용될 수 있으니, fast_completion_flag로 마킹
mask_fast_completion = (
    (course_fixed["duration"] == 0) &
    (
        (course_fixed["explored"] == 1) |
        (course_fixed["certified"] == 1)
    )
)

course_fixed["fast_completion_flag"] = 0
course_fixed.loc[mask_fast_completion, "fast_completion_flag"] = 1

print("fast completion 수:", course_fixed["fast_completion_flag"].sum())

fast completion 수: 415


### 특정 날짜(2013-11-17) 로그 제거

#### 문제 정의
- `last_event_DI = 2013-11-17`인 데이터가 약 5,600건 존재
- 해당 데이터는 다음과 같은 특징을 보임:
  - `nevents`, `ndays_act`, `nplay_video`, `nchapters` 모두 결측
  - 단일 강의(course_id)에 집중
  - 다른 데이터의 시간 분포와 비교했을 때 이례적으로 특정 날짜에 집중됨

#### 해석
- 일반적인 학습 로그라면 다양한 날짜에 분산되어 나타나야 하나,
  특정 날짜에만 집중되어 나타나는 것은 자연스러운 사용자 행동 패턴으로 보기 어려움

- 또한 활동 지표가 동시에 결측인 점을 고려할 때,
  해당 데이터는 실제 사용자 행동이 아닌
  **데이터 수집 또는 종료 과정에서 생성된 시스템 로그일 가능성이 높음**

- 특히 해당 강의가 Spring 학기임에도 불구하고
  11월에 일괄적으로 종료 로그가 기록된 점은
  데이터 수집 컷오프 또는 배치 처리 과정의 결과로 해석됨

#### 처리 방식
- 해당 날짜(`2013-11-17`)에 해당하는 데이터는
  실제 학습 행동을 반영하지 않는 것으로 판단하여 제거

#### 결론
→ 특정 날짜에 집중된 구조적 로그를 제거함으로써,
   분석 대상 데이터를 실제 사용자 행동 중심으로 정제함

In [116]:
# 날짜 분포 확인
course_fixed["last_event_DI"].value_counts().head(10)

last_event_DI
2013-11-17    5637
2013-03-13    4795
2012-08-17    4539
2012-10-15    4209
2012-10-16    3858
2013-03-14    3489
2013-02-15    3288
2013-02-14    3284
2012-08-18    3261
2013-03-12    3100
Name: count, dtype: int64

In [117]:
# 특정 날짜 필터
cutoff_date = pd.to_datetime("2013-11-17")

mask_cutoff = (
    (course_fixed["last_event_DI"] == cutoff_date)
)

print("컷오프 후보 개수:", mask_cutoff.sum())

# 필요하면 확인
course_fixed.loc[mask_cutoff, ["course_id", "nevents", "ndays_act"]].head()

# 제거
course_fixed = course_fixed.loc[~mask_cutoff].copy()

컷오프 후보 개수: 5637


### `nchapters`, `nplay_video` 조건부 결측 처리

- `nchapters`와 `nplay_video`의 결측은 모두 같은 의미를 가진다고 보기 어렵다.
- 일부 결측은 실제 활동이 없었던 경우일 수 있지만,
  일부는 로그가 관측되지 않은 결과일 가능성도 존재한다.

- 따라서 남아 있는 결측값을 일괄적으로 0으로 대체하지 않고,
  **등록 후 추가 활동 없이 이탈한 것으로 해석 가능한 경우에만**
  조건부로 0 처리하였다.

처리 기준:
- `viewed != 1`
- `last_event_DI = start_time_DI`

해석:
- 해당 조건을 만족하는 사용자는 등록 이후 별도의 학습 행동이 관측되지 않은 집단으로 보고,
  `nchapters = 0`, `nplay_video = 0`으로 처리하였다.

- 반면, 강의 접근 이력이 있거나(`viewed = 1`) 추가 학습 가능성이 남아 있는 경우에는
  결측을 임의로 0으로 바꾸지 않고 유지하였다.

→ 결측값의 의미를 구분하여 보수적으로 처리하였다.

In [118]:
# 복사본
course_conditional = course_fixed.copy()

# 조건:
# 1) viewed != 1
# 2) last_event_DI == start_time_DI
# → 등록 후 별도 활동 없이 이탈한 사용자로 해석 가능한 경우
mask_registered_only = (
    (course_conditional["viewed"] != 1) &
    (course_conditional["last_event_DI"] == course_conditional["start_time_DI"])
)

# 조건부 결측 처리 대상 컬럼
target_cols = ["nchapters", "nplay_video"]

# 해당 조건에서만 null을 0으로 대체
course_conditional.loc[mask_registered_only, target_cols] = (
    course_conditional.loc[mask_registered_only, target_cols].fillna(0)
)

# 확인
print("조건 적용 대상 수:", mask_registered_only.sum())
print()
print("처리 후 결측 개수:")
print(course_conditional[target_cols].isna().sum())
print()

print(
    course_conditional.loc[
        mask_registered_only,
        ["viewed", "start_time_DI", "last_event_DI", "nchapters", "nplay_video"]
    ].head()
)

조건 적용 대상 수: 167568

처리 후 결측 개수:
nchapters       58210
nplay_video    185071
dtype: int64

    viewed start_time_DI last_event_DI  nchapters  nplay_video
25       0    2012-09-04    2012-09-04        0.0          0.0
32       0    2012-12-23    2012-12-23        0.0          0.0
35       0    2012-09-05    2012-09-05        0.0          0.0
40       0    2012-08-27    2012-08-27        0.0          0.0
45       0    2012-09-17    2012-09-17        0.0          0.0


### age 생성

- `YoB_DI`(출생연도)와 `start_time_DI`(수강 시작일)를 이용하여 `age` 변수를 생성하였다.
- 연 단위로 계산한 값이므로, 정확한 만 나이가 아니라 분석용 근사 연령값에 해당한다.

처리 방식:
- `age = start_year - YoB_DI`
- 비현실적인 값은 결측으로 처리

→ 학습자 연령 분포를 해석하기 위한 파생 변수로 활용

### age 구간화 변수 생성

- 연속형 연령값(`age`)은 해석과 시각화의 편의를 위해 구간화하였다.

구간 기준:
- 20세 미만
- 20대
- 30대
- 40대
- 50대
- 60세 이상

→ 연령대별 학습 패턴 비교를 위한 범주형 변수(`age_group`) 생성

In [119]:
# 복사본
course_final = course_conditional.copy()

# -----------------------------------
# age 생성
# -----------------------------------
# start_time_DI에서 연도 추출
course_final["start_year"] = pd.to_datetime(course_final["start_time_DI"], errors="coerce").dt.year

# YoB_DI를 숫자로 변환
course_final["YoB"] = pd.to_numeric(course_final["YoB"], errors="coerce")

# age 계산
course_final["age"] = course_final["start_year"] - course_final["YoB"]

# 비현실적 나이는 결측 처리 (선택)
course_final.loc[(course_final["age"] < 0) | (course_final["age"] > 100), "age"] = np.nan

# -----------------------------------
# age 구간화
# -----------------------------------
bins = [0, 19, 29, 39, 49, 59, 200]
labels = ["under_20", "20s", "30s", "40s", "50s", "60_plus"]

course_final["age_group"] = pd.cut(
    course_final["age"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

### exam_flag 생성

- `grade` 값이 존재한다는 것은 최소한 평가 또는 시험 기록이 존재한다는 의미로 해석하였다.

처리 방식:
- `grade`가 결측이 아니면 `exam_flag = 1`
- `grade`가 결측이면 `exam_flag = 0`

→ 시험 또는 평가 참여 여부를 구분하는 보조 변수로 활용

In [120]:
# -----------------------------------
# exam_flag 생성
# -----------------------------------
# grade가 존재하면 시험 또는 평가 기록이 있다고 보고 1, 없으면 0
course_final["exam_flag"] = course_final["grade"].notna().astype(int)

### 범주형 결측값 처리

- `LoE_DI`, `gender`, `age_group`의 결측은 제거하지 않고 `"unknown"`으로 대체하였다.
- 이는 해당 결측이 실제 정보 부재를 의미하며,
  행 제거보다 정보 보존 측면에서 유리하다고 판단했기 때문이다.

→ 범주형 변수의 결측을 별도 범주로 유지하여 분석에 활용

In [121]:
# -----------------------------------
# 범주형 결측값 unknown 처리
# -----------------------------------
course_final["LoE_DI"] = course_final["LoE_DI"].fillna("unknown")
course_final["gender"] = course_final["gender"].fillna("unknown")
course_final["age_group"] = course_final["age_group"].astype("object").fillna("unknown")

In [122]:
# 확인
print(course_final[["YoB", "start_time_DI", "age", "age_group", "grade", "exam_flag", "LoE_DI", "gender"]].head())

print("\nage_group 분포")
print(course_final["age_group"].value_counts(dropna=False))

print("\nexam_flag 분포")
print(course_final["exam_flag"].value_counts(dropna=False))

    YoB start_time_DI  age age_group grade  exam_flag   LoE_DI   gender
5   NaN    2012-09-17  NaN   unknown     0          1  unknown  unknown
7   NaN    2013-01-01  NaN   unknown     0          1  unknown  unknown
8   NaN    2013-02-18  NaN   unknown     0          1  unknown  unknown
10  NaN    2013-02-23  NaN   unknown     0          1  unknown  unknown
11  NaN    2013-06-17  NaN   unknown     0          1  unknown  unknown

age_group 분포
age_group
20s         269213
30s          86206
unknown      81054
under_20     56531
40s          25461
50s          10819
60_plus       4709
Name: count, dtype: int64

exam_flag 분포
exam_flag
1    491268
0     42725
Name: count, dtype: int64


In [123]:
course_final.to_csv('course_final.csv', index=False)